# Paso 03 -- Escalon 3: semana completa, metricas agregadas

Corre el entorno (`SimuladorBarcosBergen`, mismo modelo de colas por par y
misma politica coordinada `asignar_flota` que los escalones 1 y 2) sobre
**7 dias independientes** (lunes..domingo, `output/escalon3/grupos_semana.csv`
del paso 00), uno por uno, y junta los resultados con
`metricas.combinar_corridas` para sacar las mismas metricas detalladas que
ya existen para escalon 1/2, pero agregadas sobre la semana entera.

**Por que 7 corridas independientes y no una unica corrida continua de
10080 minutos:** cada dia es su propio episodio (su propio reloj
`hora_inicio_min..hora_fin_min`), exactamente como escalon 1 y escalon 2 ya
lo hacen -- no hace falta inventar que hacen los barcos en la madrugada
(00-06h, sin demanda) si el reloj no se resetea entre dias. `env.py` no se
toca; `metricas.combinar_corridas` es la unica pieza nueva, y solo SUMA
resultados de corridas ya terminadas (ver `simulacion/src/metricas.py`).

**Sin animacion ni inspector** (igual que escalon 2): son 7 corridas de
hasta 540 pasos cada una, ya se verifico a ojo en escalon 1.


In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
BB_DIR = (BASE_DIR / "../bergen-boats").resolve()
sys.path.insert(0, str(BASE_DIR / "src"))

from env import SimuladorBarcosBergen
from politica_base import politica_base, asignar_flota
import metricas as met
import visualizacion as viz

cfg_sim = yaml.safe_load(open(BASE_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))

nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

OUT_ESC = BASE_DIR / "output" / "escalon3"
OUT_ESC.mkdir(parents=True, exist_ok=True)
grupos_semana = pd.read_csv(OUT_ESC / "grupos_semana.csv")

cfg_escalon = cfg_sim["escalones"]["escalon_3"]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
NOMBRES_DIA = ["lunes", "martes", "miercoles", "jueves", "viernes", "sabado", "domingo"]


def construir_env(dia_idx, grupos_dia):
    return SimuladorBarcosBergen(
        grupos_df=grupos_dia, matriz_tiempos=matriz_tiempos, nodos=nodos,
        num_barcos=cfg_escalon["num_barcos"], capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"],
        nodo_inicial=cfg_bb["flota"]["nodo_inicial"], paso_tiempo_min=cfg_sim["paso_tiempo_min"],
        hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_sim["recompensa"], unidad_demanda=cfg_sim["unidad_demanda"],
        dia_semana=dia_idx,
    )


n_pasos_por_dia = int((hora_fin_min - hora_ini_min) / cfg_sim["paso_tiempo_min"])
print("Nodos:", nodos)
print(f"Barcos: {cfg_escalon['num_barcos']}, capacidad: {cfg_bb['flota']['capacidad_pasajeros']}")
print(f"Horizonte por dia: {hora_ini_min}-{hora_fin_min} min -> {n_pasos_por_dia} pasos/dia x 7 dias")
print(f"Demanda semana: {len(grupos_semana)} grupos, {grupos_semana['tamano_grupo'].sum()} personas")


Nodos: ['kleppesto', 'laksevag', 'bryggen', 'sandviken']
Barcos: 3, capacidad: 20
Horizonte por dia: 360-1440 min -> 540 pasos/dia x 7 dias
Demanda semana: 620 grupos, 5429 personas


## Corrida completa (7 dias, uno por uno)

In [2]:
def correr_un_dia(dia_idx):
    grupos_dia = grupos_semana[grupos_semana["dia"] == dia_idx].reset_index(drop=True)
    env = construir_env(dia_idx, grupos_dia)
    obs, info = env.reset(seed=cfg_sim["semilla"])
    reward_total = 0.0
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_sim["recompensa"])
        accion = np.array([
            env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0
            for b in estado.barcos
        ])
        obs, r, terminated, truncated, info = env.step(accion)
        reward_total += r
        if truncated or terminated:
            break
    return env, reward_total


envs_semana = []
reward_total_semana = 0.0
for dia_idx in range(7):
    env_dia, reward_dia = correr_un_dia(dia_idx)
    envs_semana.append(env_dia)
    reward_total_semana += reward_dia
    tipo_dia = "fin de semana" if dia_idx >= 5 else "entre semana"
    print(f"{NOMBRES_DIA[dia_idx]:>10} ({tipo_dia}): {env_dia.total_unidades_generadas:4d} personas, "
          f"recompensa {reward_dia:9.2f}")

print(f"\n7 dias simulados. Recompensa total de la semana: {reward_total_semana:.2f}")


     lunes (entre semana):  987 personas, recompensa   -631.28


    martes (entre semana): 1013 personas, recompensa  -1230.91


 miercoles (entre semana):  756 personas, recompensa   -675.59


    jueves (entre semana): 1121 personas, recompensa  -1748.20


   viernes (entre semana):  902 personas, recompensa   -762.68


    sabado (fin de semana):  326 personas, recompensa    -58.82


   domingo (fin de semana):  324 personas, recompensa   -130.14

7 dias simulados. Recompensa total de la semana: -5237.62


## Combinar los 7 dias y verificar conservacion de personas

In [3]:
corrida_semana = met.combinar_corridas(envs_semana)

conservacion = met.verificar_conservacion(corrida_semana)
for k, v in conservacion.items():
    print(f"{k}: {v}")
assert conservacion["cuadra"], "La conservacion de personas no cuadra -- hay un bug que exponer, no esconder."
print("\nOK: todas las personas generadas en la semana quedaron contabilizadas (atendidas + esperando al final + a bordo al final). Nadie se pierde -- ver simulacion/README.md.")


generadas: 5429
atendidas: 5406
esperando_al_final: 0
a_bordo_al_final: 23
suma: 5429
cuadra: True

OK: todas las personas generadas en la semana quedaron contabilizadas (atendidas + esperando al final + a bordo al final). Nadie se pierde -- ver simulacion/README.md.


## Metricas detalladas (agregadas sobre los 7 dias)

In [4]:
reporte = met.reporte_completo(corrida_semana)

print("=== Globales (semana) ===")
for k, v in reporte["globales"].items():
    print(f"  {k}: {v:.2f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n=== Por par (solo pares con demanda) ===")
df_par = reporte["por_par"]
display(df_par[df_par["generadas"] > 0].round(1))


=== Globales (semana) ===
  unidades_generadas: 5429
  unidades_atendidas: 5406
  unidades_sin_atender_al_final: 23
  pct_atendidas: 99.58
  espera_media_min: 10.04
  sistema_medio_min: 19.75
  sistema_maximo_min: 57.12

=== Por par (solo pares con demanda) ===


,par,generadas,atendidas,sin_atender_al_final,pct_atendidas,espera_media_min,espera_maxima_min,viaje_medio_min,sistema_medio_min,sistema_maximo_min
0,kleppesto->laksevag,315,315,0,100.0,11.0,26.7,6.0,17.0,32.7
1,kleppesto->bryggen,786,780,6,99.2,8.4,26.2,12.0,20.4,38.2
2,kleppesto->sandviken,109,109,0,100.0,11.1,25.7,10.0,21.1,35.7
3,laksevag->kleppesto,385,385,0,100.0,10.6,51.1,6.0,16.6,57.1
4,laksevag->bryggen,719,712,7,99.0,11.4,34.7,10.0,21.4,44.7
5,laksevag->sandviken,194,194,0,100.0,12.6,40.8,10.0,22.6,50.8
6,bryggen->kleppesto,944,944,0,100.0,9.1,29.8,12.0,21.1,41.8
7,bryggen->laksevag,615,605,10,98.4,10.1,31.8,10.0,20.1,41.8
8,bryggen->sandviken,567,567,0,100.0,8.8,27.8,8.0,16.8,35.8
9,sandviken->kleppesto,174,174,0,100.0,10.2,23.8,10.0,20.2,33.8


In [5]:
print("=== Por barco ===")
display(reporte["por_barco"].round(1))

print("\n=== Por usuario (percentiles, minutos) ===")
for k, v in reporte["por_usuario"].items():
    print(f"  {k}: {v}")

print("\n=== Sin atender al final (backlog, no perdidas) ===")
display(reporte["sin_atender_al_final"])

reporte["por_par"].to_csv(OUT_ESC / "metricas_por_par.csv", index=False)
reporte["por_barco"].to_csv(OUT_ESC / "metricas_por_barco.csv", index=False)


=== Por barco ===


,barco_id,movimientos,tiempo_navegado_min,ocupacion_media,ocupacion_maxima,pct_ocioso,ruta_mas_frecuente
0,barco_0,438,3280,2.0,20,56.7,bryggen->kleppesto (59x)
1,barco_1,413,3042,1.9,20,59.8,bryggen->laksevag (54x)
2,barco_2,369,2790,1.5,20,63.2,kleppesto->bryggen (51x)



=== Por usuario (percentiles, minutos) ===
  espera_min: {'p50': 9.060000000000002, 'p90': 21.389999999999986, 'p95': 25.139999999999986, 'max': 51.120000000000005}
  viaje_min: {'p50': 10.0, 'p90': 12.0, 'p95': 12.0, 'max': 12.0}
  sistema_min: {'p50': 18.769999999999982, 'p90': 31.25, 'p95': 35.319999999999936, 'max': 57.120000000000005}

=== Sin atender al final (backlog, no perdidas) ===


,par,sin_atender
0,bryggen->laksevag,10
1,laksevag->bryggen,7
2,kleppesto->bryggen,6


## Logs crudos (para auditar sin volver a correr nada)

In [6]:
pd.DataFrame(corrida_semana.log_eventos).to_csv(OUT_ESC / "log_eventos.csv", index=False)
pd.DataFrame(corrida_semana.log_recompensa).to_csv(OUT_ESC / "log_recompensa.csv", index=False)
print(f"{len(corrida_semana.log_eventos)} eventos guardados en log_eventos.csv")
print(f"{len(corrida_semana.log_recompensa)} pasos de recompensa guardados en log_recompensa.csv")


18843 eventos guardados en log_eventos.csv
3780 pasos de recompensa guardados en log_recompensa.csv


## Graficas

In [7]:
fig = viz.graficar_perfil_espera(corrida_semana)
fig.write_html(OUT_ESC / "wait_profile.html")
fig.show()


In [8]:
fig = viz.graficar_ocupacion_flota(corrida_semana)
fig.write_html(OUT_ESC / "fleet_occupancy.html")
fig.show()


In [9]:
fig = viz.graficar_heatmap_cumplimiento(corrida_semana)
fig.write_html(OUT_ESC / "pct_served_heatmap.html")
fig.show()


In [10]:
fig = viz.graficar_desglose_recompensa(corrida_semana)
fig.write_html(OUT_ESC / "reward_breakdown.html")
fig.show()


In [11]:
fig = viz.graficar_sin_atender_al_final(corrida_semana)
fig.write_html(OUT_ESC / "backlog_by_pair.html")
fig.show()


## Verificacion de reproducibilidad

In [12]:
sys.path.insert(0, str((BASE_DIR / "../demand/src").resolve()))
import llegadas as demand_llegadas
import masas as demand_masas

DEMAND_DIR = (BASE_DIR / "../demand").resolve()
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")
resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
bb_cfg_path2 = (DEMAND_DIR / cfg_demand["fuentes_externas"]["nodos_config"]).resolve()
cfg_bb2 = yaml.safe_load(open(bb_cfg_path2, encoding="utf-8"))
conexiones_fuertes = cfg_bb2["garantia"]["conexiones_fuertes"]


def generar_semana_con_semilla(semilla):
    cfg_mod = dict(cfg_demand)
    cfg_mod["demanda"] = dict(cfg_demand["demanda"])
    cfg_mod["demanda"]["porcentaje_poblacion_dia"] = cfg_escalon["porcentaje_poblacion_dia"]
    grupos = demand_llegadas.generar_llegadas_semana(
        cfg_mod, intensidad_od, conexiones_fuertes, poblacion_total_zonas, semilla=semilla,
    )
    hora_ini, hora_fin = cfg_escalon["horas"]
    return grupos[(grupos["hora"] >= hora_ini) & (grupos["hora"] < hora_fin)].reset_index(drop=True)


def correr_semana(grupos_sem):
    envs = []
    for dia_idx in range(7):
        grupos_dia = grupos_sem[grupos_sem["dia"] == dia_idx].reset_index(drop=True)
        env = construir_env(dia_idx, grupos_dia)
        obs, info = env.reset(seed=cfg_sim["semilla"])
        total = 0.0
        while True:
            estado = info["_estado_obj"]
            libres = [b for b in estado.barcos if b.libre]
            decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_sim["recompensa"])
            accion = np.array([
                env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0
                for b in estado.barcos
            ])
            obs, r, terminated, truncated, info = env.step(accion)
            total += r
            if truncated or terminated:
                break
        envs.append(env)
    corrida = met.combinar_corridas(envs)
    return round(total, 6), len(corrida.atendidas_historico), sum(len(c) for c in corrida.colas.values())


grupos_a = generar_semana_con_semilla(cfg_sim["semilla"])
grupos_b = generar_semana_con_semilla(cfg_sim["semilla"])
grupos_c = generar_semana_con_semilla(cfg_sim["semilla"] + 1)

resultado_a = correr_semana(grupos_a)
resultado_b = correr_semana(grupos_b)
resultado_c = correr_semana(grupos_c)

print(f"Semilla {cfg_sim['semilla']}, generacion a: {resultado_a}")
print(f"Semilla {cfg_sim['semilla']}, generacion b: {resultado_b}")
print(f"Semilla {cfg_sim['semilla'] + 1}:               {resultado_c}")
print("Misma semilla de demanda -> misma corrida completa:", resultado_a == resultado_b)
print("Semilla distinta -> corrida distinta:", resultado_a != resultado_c)


Semilla 42, generacion a: (-130.141732, 5406, 0)
Semilla 42, generacion b: (-130.141732, 5406, 0)
Semilla 43:               (-105.445683, 5250, 19)
Misma semilla de demanda -> misma corrida completa: True
Semilla distinta -> corrida distinta: True
